# benchmarking_minatar

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"
REFERENCE_CONFIG = Path("../minatar_deterministic/config.py")

In [ ]:
# Load Agent0's stored runs per game, and the DQN runs of minatar_deterministic
# as the reference.
AGENT0 = "Agent0"
DQN = "DQN"
COMPONENT = "agent0_minatar"
component = EXPERIMENT.component(COMPONENT)
GAMES = component.sweep["ENV_HYPERS.GAME"]

spec = importlib.util.spec_from_file_location("reference_config", REFERENCE_CONFIG)
reference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reference)
STORES = {
    AGENT0: (EXPERIMENT, COMPONENT),
    DQN: (reference.EXPERIMENT, "dqn_minatar"),
}
AGENTS = list(STORES)

results = {}
for agent, (experiment, name) in STORES.items():
    for game in GAMES:
        runs = load_results(experiment, name, where={"ENV_HYPERS.GAME": game})
        if runs is not None:
            results[agent, game] = runs

In [ ]:
# Compute each agent's mean return over steps per game with a 95% bootstrap CI.
curves = {}
for (agent, game), runs in results.items():
    steps, returns = subsample(return_curve(runs.reward, runs.done))
    curves[agent, game] = (steps, mean_ci(returns))

In [ ]:
# Style for the learning-curve plots. Every piece of text shares one size.
TOTAL_STEPS = component.config.AGENT_HYPERS.TOTAL_TIMESTEPS
FONT_SIZE = 32
STYLE = {
    "figure.figsize": (32, 8),
    "font.family": "Times New Roman",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
}
BAND_ALPHA = 0.2
DQN_COLOR = "tab:blue"
AGENT0_COLOR = "tab:red"
COLORS = {AGENT0: AGENT0_COLOR, DQN: DQN_COLOR}
# Only the left panel carries the legend, since the panels share colors.
LEGEND = {
    "loc": "lower right",
    "bbox_to_anchor": (1.04, -0.04),
    "handlelength": 1,
    "handletextpad": 0.4,
}
LEGEND_MARKER = {"marker": "o", "linestyle": "none", "markersize": 14}
BOX_ASPECT = 1
# Ticks sit only at the axis ends, so the labels can tuck in between them.
LABEL_PAD = -20
XLABEL = "Time Steps"
XTICKS = [0, TOTAL_STEPS]
XTICK_LABELS = ["0", f"{TOTAL_STEPS // 1_000_000}M"]
YLABEL = "Return"
YTICK_STEP = 10


def title(game):
    return game.replace("_", " ").title()


def top_of_axis(game):
    # The axis ends at the next multiple of YTICK_STEP above every curve drawn.
    highest = max(
        np.nanmax(curves[agent, game][1].high)
        for agent in AGENTS
        if (agent, game) in curves
    )
    return int(np.ceil(highest / YTICK_STEP) * YTICK_STEP)

In [ ]:
# Plot every game's panel side by side. Each panel has its own y-axis, so each
# is labelled. Agent0 is drawn over the DQN reference where runs are stored.
with plt.rc_context(STYLE):
    fig, axes = plt.subplots(1, len(GAMES))
    for ax, game in zip(axes, GAMES, strict=True):
        ax.set_box_aspect(BOX_ASPECT)
        for agent in AGENTS:
            if (agent, game) not in curves:
                continue
            steps, (mean, low, high) = curves[agent, game]
            color = COLORS[agent]
            ax.fill_between(steps, low, high, color=color, alpha=BAND_ALPHA)
            ax.plot(steps, mean, color=color)
        top = top_of_axis(game)
        ax.set_title(title(game))
        ax.set_xlabel(XLABEL, labelpad=LABEL_PAD)
        ax.set_xticks(XTICKS, XTICK_LABELS)
        ax.set_yticks([0, top], ["0", str(top)])
        ax.set_ylim(0, top)
        ax.set_ylabel(YLABEL, labelpad=LABEL_PAD)
    fig.tight_layout()
    drawn = [agent for agent in AGENTS if any((agent, g) in curves for g in GAMES)]
    handles = [Line2D([], [], color=COLORS[a], **LEGEND_MARKER) for a in drawn]
    axes[0].legend(handles, drawn, **LEGEND)
figures = {"grid": fig}
plt.show()

In [ ]:
# Save every plotted figure as PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    for ext in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=200)
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")